# SpeakAI-Eval — Inference (Run Only)

Notebook này chỉ cần gọi lên và chạy. Đảm bảo bạn đã chạy **Setup Notebook** trước đó.
Mã nguồn, models và HF cache sẽ được đọc thẳng từ Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Cài đặt thư viện
import subprocess, sys, os
pkgs = [
    'torch>=2.1.0', 'torchaudio>=2.1.0', 'torch-geometric>=2.4.0',
    'transformers>=4.36.0', 'peft>=0.7.0', 'pyyaml>=6.0.1',
    'numpy>=1.24.0', 'soundfile>=0.12.1', 'nltk>=3.8.1',
    'noisereduce>=3.0.0', 'python-dotenv>=1.0.0',
    'speechbrain>=1.0.0', 'huggingface_hub>=0.23.0', 'silero-vad>=6.2.1',
    'accelerate>=0.26.0', 'fastapi', 'uvicorn', 'python-multipart',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs, check=True)
print('Libraries installed!')
if not os.path.exists('/usr/local/bin/cloudflared'):
    print('Downloading cloudflared...')
    subprocess.run('wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared', shell=True, check=True)
    subprocess.run('chmod +x /usr/local/bin/cloudflared', shell=True, check=True)


---
## Khởi tạo Pipeline từ Drive

In [ ]:
import sys, os
TARGET_DIR = '/content/drive/MyDrive/SpeakAI-Eval'
sys.path.insert(0, TARGET_DIR)
sys.path.insert(0, f'{TARGET_DIR}/ecapa-diarize')
os.chdir(TARGET_DIR)

# Lưu cache HuggingFace vào Drive để không phải tải lại LLM/Whisper
os.environ['HF_HOME'] = f'{TARGET_DIR}/hf_cache'

# VÁ LỖI GOOGLE DRIVE READ-ONLY (monkey-patch)
import ecapa_diarize.embedding
import tempfile
original_init = ecapa_diarize.embedding.EcapaEmbedder.__init__
def patched_init(self, device="cpu"):
    from speechbrain.inference.speaker import SpeakerRecognition
    from speechbrain.utils.fetching import LocalStrategy
    from ecapa_diarize.paths import ECAPA_DIR
    cache_dir = os.path.join(tempfile.gettempdir(), "ecapa_cache")
    os.makedirs(cache_dir, exist_ok=True)
    self.device = device
    self._model = SpeakerRecognition.from_hparams(
        source=str(ECAPA_DIR),
        savedir=cache_dir,
        run_opts={"device": device},
        local_strategy=LocalStrategy.COPY,
    )
ecapa_diarize.embedding.EcapaEmbedder.__init__ = patched_init

import torch
from infer.pipeline import SpeakingPipeline
from transformers import AutoModelForCausalLM, AutoTokenizer

print('Loading Pipeline models on GPU...')
pipeline = SpeakingPipeline(device='cuda')
print('Pipeline ready!')

model_name = '/content/drive/MyDrive/SpeakAI-Eval/pretrained_models/Qwen2.5-3B-Instruct'
print(f'Loading LLM from Drive: {model_name}')
tokenizer = AutoTokenizer.from_pretrained(model_name)
llm_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map='cuda'
)
print(f'LLM Ready on {llm_model.device}!')


---
## Hàm Sinh Feedback Tổng Hợp bằng Qwen LLM

In [ ]:
def generate_overall_feedback(result):
    student_sents = result.get('student', {}).get('sentences', [])
    if not student_sents:
        return 'Không có dữ liệu học viên để đánh giá.'
    
    total_acc = sum(s.get('scores', {}).get('accuracy', 0) for s in student_sents) / len(student_sents)
    total_flu = sum(s.get('scores', {}).get('fluency', 0) for s in student_sents) / len(student_sents)
    total_pro = sum(s.get('scores', {}).get('prosodic', 0) for s in student_sents) / len(student_sents)
    overall_total = (total_acc + total_flu + total_pro) / 3
    
    dialogue_turns = result.get('dialogue', {}).get('turns', [])
    conversation_text = ''
    bad_words_all = []
    bad_ph_all = []
    
    for turn in dialogue_turns:
        speaker = turn.get('role', '').upper()
        transcript = turn.get('transcript', '')
        if speaker == 'TEACHER':
            conversation_text += f'Giáo viên: {transcript}'
        elif speaker == 'STUDENT':
            turn_score = turn.get('scores', {}).get('accuracy', 0)
            conversation_text += f'Học viên: {transcript} (Điểm phát âm: {turn_score:.1f}/10)'
            errors = turn.get('errors', {})
            bad_words = [w for w in errors.get('words', []) if w.get('score', 10) < 7.0]
            bad_ph = [p for p in errors.get('phonemes', []) if p.get('score', 10) < 7.0]
            bad_words_all.extend([f"{w['word']} ({w.get('score',0):.1f})" for w in bad_words])
            bad_ph_all.extend([f"{p['phoneme']} ({p.get('score',0):.1f})" for p in bad_ph])
    
    bad_words_str = ', '.join(list(dict.fromkeys(bad_words_all))[:15]) or 'Không có'
    bad_ph_str = ', '.join(list(dict.fromkeys(bad_ph_all))[:15]) or 'Không có'
    
    prompt = f"""Bạn là một giáo viên chuyên đánh giá phát âm và giao tiếp tiếng Anh.\nDưới đây là đoạn hội thoại giữa Giáo viên và Học viên, cùng với kết quả phân tích phát âm của Học viên. Hãy viết một bài nhận xét TỔNG HỢP (Overall Feedback) thật chi tiết, rõ ràng và có cấu trúc dễ đọc.\n\n--- ĐOẠN HỘI THOẠI ---\n{conversation_text}\n--- ĐIỂM SỐ TRUNG BÌNH CỦA HỌC VIÊN (Thang 10) ---\nTổng quan: {overall_total:.1f}\nChính xác (Accuracy): {total_acc:.1f}\nTrôi chảy (Fluency): {total_flu:.1f}\nNgữ điệu (Prosody): {total_pro:.1f}\n\n--- LỖI PHÁT ÂM ĐÁNG CHÚ Ý CẦN SỬA ---\nTừ phát âm sai nhiều: {bad_words_str}\nÂm vị (Phoneme) sai nhiều: {bad_ph_str}\n\nYêu cầu nhận xét (bằng tiếng Việt, định dạng Markdown đẹp, rõ ràng):\n1. Đánh giá chung: Học viên làm tốt ở đâu (khen ngợi), giao tiếp có tự nhiên và đúng ngữ cảnh không? Ngữ pháp sử dụng có đúng không? (LƯU Ý: Chỉ liệt kê lỗi sai ngữ pháp, KHÔNG viết lại những câu đã chính xác để tránh dài dòng).\n2. Điểm cần khắc phục: Giải thích thật rõ ràng các lỗi phát âm (từ/âm vị cụ thể) và hướng dẫn cách sửa chi tiết.\n3. Lời khuyên & Động viên: Đề xuất cách luyện tập để cải thiện.\nKhông nhắc đến 'Completeness'.\n"""
    messages = [
        {'role': 'system', 'content': 'Bạn là giáo viên tiếng Anh tận tâm, chuyên môn cao.'},
        {'role': 'user', 'content': prompt}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer([text], return_tensors='pt').to(llm_model.device)
    
    generated_ids = llm_model.generate(**model_inputs, max_new_tokens=512)
    generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)]
    
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer([text], return_tensors='pt').to(llm_model.device)
    
    generated_ids = llm_model.generate(**model_inputs, max_new_tokens=512)
    generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)]
    
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return response


---
## Khởi chạy Backend API (FastAPI + Cloudflare Tunnel)

In [ ]:
from fastapi import FastAPI, UploadFile, File, Form, BackgroundTasks
import uuid
from fastapi.middleware.cors import CORSMiddleware
from fastapi.staticfiles import StaticFiles
from fastapi.responses import JSONResponse
import uvicorn
import shutil
import json
import numpy as np
import subprocess
import time
import re
import os

# 1. Khởi động Cloudflare Tunnel
def start_cloudflare_tunnel(port=8000):
    print('Starting Cloudflare Tunnel...')
    cmd = f'cloudflared tunnel --url http://127.0.0.1:{port}'
    process = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    url = None
    for _ in range(20):
        line = process.stdout.readline()
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if match:
            url = match.group(0)
            break
        time.sleep(0.5)
    return url

PUBLIC_URL = start_cloudflare_tunnel(8000)
print('' + '='*80)
print(f'🚀 API IS LIVE AT: {PUBLIC_URL}')
print('=> COPY LINK NÀY VÀ DÁN VÀO CẤU HÌNH TRÊN WEBSITE CỦA BẠN!')
print('='*80 + '')

# 2. Khởi tạo FastAPI App
app = FastAPI()
app.add_middleware(
    CORSMiddleware,
    allow_origins=['*'],
    allow_credentials=True,
    allow_methods=['*'],
    allow_headers=['*'],
)

# Phục vụ file audio tĩnh từ Colab để Website có thể nghe lại
os.makedirs('/tmp/SpeakAI_Audio', exist_ok=True)
app.mount('/audio', StaticFiles(directory='/tmp/SpeakAI_Audio'), name='audio')

tasks = {}

@app.post('/assess_start')
def assess_start_api(background_tasks: BackgroundTasks, audio: UploadFile = File(...), teacher_embeddings_json: str = Form(...), student_embeddings_json: str = Form(...), score_teacher: bool = Form(False)):
    try:
        task_id = str(uuid.uuid4())
        tasks[task_id] = {'status': 'processing', 'step': 'Đang tải file âm thanh lên server...', 'result': None, 'llm_feedback': None}
        
        conv_path = f'/tmp/SpeakAI_Audio/{task_id}_{audio.filename}'
        with open(conv_path, 'wb') as f:
            shutil.copyfileobj(audio.file, f)
        
        background_tasks.add_task(process_assessment, task_id, conv_path, teacher_embeddings_json, student_embeddings_json, score_teacher)
        return JSONResponse({'success': True, 'task_id': task_id})
    except Exception as e:
        return JSONResponse({'success': False, 'error': str(e)}, status_code=500)

def process_assessment(task_id, conv_path, teacher_embeddings_json, student_embeddings_json, score_teacher):
    try:
        tasks[task_id]['step'] = 'Đang phân tích embeddings...'
        
        # Parse Embeddings của Giáo viên
        t_emb_list = json.loads(teacher_embeddings_json)
        teacher_emb = np.array(t_emb_list, dtype=np.float32)
        if len(teacher_emb.shape) == 2:
            teacher_emb = np.mean(teacher_emb, axis=0)
        teacher_emb /= np.linalg.norm(teacher_emb)

        # Parse Embeddings của Học viên
        s_emb_list = json.loads(student_embeddings_json)
        student_emb = np.array(s_emb_list, dtype=np.float32)
        if len(student_emb.shape) == 2:
            student_emb = np.mean(student_emb, axis=0)
        student_emb /= np.linalg.norm(student_emb)
        
        # Gọi Pipeline
        tasks[task_id]['step'] = 'Đang tách lời (Diarization) & Phân tích phát âm...'
        raw_result = pipeline.assess_conversation(
            conv_path,
            teacher_embedding=teacher_emb,
            student_embedding=student_emb,
            score_teacher=score_teacher
        )
        
        # Trích xuất file tổng hợp
        diar = raw_result.get('diarization', {})
        if raw_result.get('teacher') and diar.get('teacher'):
            raw_result['teacher']['full_audio'] = str(diar['teacher'])
        if raw_result.get('student') and diar.get('student'):
            raw_result['student']['full_audio'] = str(diar['student'])

        # Gọi LLM Feedback
        tasks[task_id]['step'] = 'Đang gọi LLM Qwen tạo Feedback...'
        llm_feedback = generate_overall_feedback(raw_result)
        
        # Chuyển đổi đường dẫn file cục bộ thành Public URL
        def convert_paths_to_urls(node):
            if isinstance(node, dict):
                for k, v in node.items():
                    if (k == 'audio' or k == 'full_audio') and isinstance(v, str) and v.startswith('/tmp/SpeakAI_Audio/'):
                        rel_path = v.replace('/tmp/SpeakAI_Audio/', '')
                        node[k] = f'{PUBLIC_URL}/audio/{rel_path}'
                    else:
                        convert_paths_to_urls(v)
            elif isinstance(node, list):
                for item in node:
                    convert_paths_to_urls(item)
                    
        convert_paths_to_urls(raw_result)
        
        # Xóa file audio tạm
        if os.path.exists(conv_path):
            os.remove(conv_path)
            
        tasks[task_id]['result'] = raw_result
        tasks[task_id]['llm_feedback'] = llm_feedback
        tasks[task_id]['status'] = 'completed'
    except Exception as e:
        print(f'API Error in task {task_id}: {e}')
        import traceback
        traceback.print_exc()
        tasks[task_id]['status'] = 'error'
        tasks[task_id]['error'] = str(e)

@app.get('/assess_status/{task_id}')
def assess_status(task_id: str):
    if task_id not in tasks:
        return JSONResponse({'success': False, 'error': 'Task not found'}, status_code=404)
    return JSONResponse({'success': True, 'data': tasks[task_id]})

# Khởi chạy Uvicorn
config = uvicorn.Config(app, host='0.0.0.0', port=8000)
server = uvicorn.Server(config)
await server.serve()
